In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

STORAGE_ACCOUNT = "vjretailflow01"
STORAGE_KEY = dbutils.secrets.get(scope="retailflow", key="adls-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"

print("Setup complete")

In [0]:
payments_bronze = spark.read.format("delta") \
    .load(f"{BRONZE_PATH}/payments/")

payments_bronze.printSchema()
display(payments_bronze.limit(3))

In [0]:
def transform_payments(df):
    df_clean = df \
        .filter(F.col("order_id").isNotNull()) \
        .filter(F.col("payment_value") > 0) \
        .withColumn("order_id", F.col("order_id").cast(T.StringType())) \
        .withColumn("payment_sequential", F.col("payment_sequential").cast(T.IntegerType())) \
        .withColumn("payment_type", F.col("payment_type").cast(T.StringType())) \
        .withColumn("payment_installments", F.col("payment_installments").cast(T.IntegerType())) \
        .withColumn("payment_value", F.round(F.col("payment_value"),2)) \
        .drop("_ingested_at", "_source_file", "_ingestion_date")
    
    df_agg = df_clean.groupBy("order_id").agg(
        F.round(F.sum("payment_value"),2).alias("total_payment_value"),
        F.count("payment_sequential").alias("payment_count"),
        F.max("payment_installments").alias("max_payment_installments"),
        F.collect_set("payment_type").alias("payment_types")
    ) \
    .withColumn("is_multi_payment", 
        F.when(F.col("payment_count") > 1, True).otherwise(False)            
    ) \
    .withColumn("_silver_processed_at", F.current_timestamp())
    
    return df_agg

payments_silver = transform_payments(payments_bronze)
display(payments_silver.limit(3))

In [0]:
print(f"Total payment rows: {payments_bronze.count():,}")
print(f"Aggregated order rows: {payments_silver.count():,}")
print(f"Multi-payment_orders: {payments_silver.filter(F.col('is_multi_payment')).count():,}")

print(f"\nPayment type breakdown:")
display(payments_bronze.groupBy("payment_type") \
    .count()
    .orderBy(F.desc("count")))

print(f"\nPayment value summary:")
display(payments_silver.agg(
    F.round(F.sum("total_payment_value"),2).alias("total_revenue"),
    F.round(F.avg("total_payment_value"),2).alias("avg_payment"),
    F.max("max_payment_installments").alias("max_installments")
))

payments_silver.write.format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER_PATH}/payments")

print("Payments silver table written successfully!")